In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1JBADzX_K62w5Ibksr4P24pVHUDoTPOxT", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))


# Notebook 1: Activation Checkpointing (Gradient Checkpointing)

**Vizuara AI Pods | GPU Programming Course | Pod 3: Recomputation, Accumulation, and Data Parallelism**

---

In this notebook, we will build a deep understanding of **activation checkpointing** (also called gradient checkpointing) -- the technique of trading compute for memory by discarding and recomputing activations during backpropagation.

By the end of this notebook, you will:

1. Understand why activations dominate GPU memory during training
2. Implement activation checkpointing from scratch (no libraries)
3. Use PyTorch's built-in `torch.utils.checkpoint` API
4. Measure the actual memory savings and compute overhead
5. Compare full checkpointing vs selective checkpointing

**Prerequisites:** Basic PyTorch knowledge (forward/backward pass, autograd).

**Estimated time:** 40-50 minutes

**Runtime:** Make sure you are using a GPU runtime in Colab: Runtime > Change runtime type > T4 GPU

In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Setup

In [ ]:
!pip install -q torch matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
import matplotlib.pyplot as plt
import numpy as np
import time
import gc

assert torch.cuda.is_available(), "No GPU found! Please enable GPU in Runtime > Change runtime type."
device = torch.device('cuda')
print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
#@title 🎧 Code Walkthrough: Memory Helpers
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_memory_helpers.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def get_memory_mb():
    """Return current GPU memory allocated in MB."""
    return torch.cuda.memory_allocated() / (1024 ** 2)

def get_peak_memory_mb():
    """Return peak GPU memory allocated in MB."""
    return torch.cuda.max_memory_allocated() / (1024 ** 2)

def reset_memory_stats():
    """Reset peak memory tracking and clear cache."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(f"Initial memory: {get_memory_mb():.1f} MB")

In [ ]:
#@title 🎧 Listen: Activations Dominate
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_activations_dominate.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 1: Why Activations Dominate Memory

During the forward pass of a neural network, every layer produces an output -- an **activation**. PyTorch's autograd engine stores these activations because they are needed during the backward pass to compute gradients.

Let's build a simple multi-layer network and measure how much memory the activations consume.

In [ ]:
#@title 🎧 Code Walkthrough: Measure Activation Memory
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_measure_activation_memory.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SimpleDeepNet(nn.Module):
    """A simple deep network to demonstrate activation memory growth."""
    def __init__(self, hidden_dim, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = F.gelu(norm(layer(x)))
        return x

# Let's measure how activation memory scales with the number of layers
hidden_dim = 2048
batch_size = 32
seq_len = 512

layer_counts = [4, 8, 12, 16, 20, 24]
activation_memories = []

for num_layers in layer_counts:
    reset_memory_stats()

    model = SimpleDeepNet(hidden_dim, num_layers).to(device)
    x = torch.randn(batch_size, seq_len, hidden_dim, device=device)

    mem_before = get_memory_mb()
    output = model(x)
    loss = output.sum()
    mem_after_forward = get_memory_mb()

    activation_mem = mem_after_forward - mem_before
    activation_memories.append(activation_mem)

    print(f"Layers: {num_layers:2d} | Activation memory: {activation_mem:8.1f} MB")

    # Clean up
    del model, x, output, loss
    reset_memory_stats()

# Calculate expected memory per activation
expected_per_layer = batch_size * seq_len * hidden_dim * 4 / (1024**2)  # float32
print(f"\nExpected memory per layer activation: {expected_per_layer:.1f} MB")
print(f"Actual memory growth per layer: ~{(activation_memories[-1] - activation_memories[0]) / (layer_counts[-1] - layer_counts[0]):.1f} MB")
print("\nActivation memory grows linearly with the number of layers!")

In [ ]:
#@title 🎧 What to Look For: Activation Memory Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_activation_memory_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the linear growth
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(layer_counts, activation_memories, 'o-', color='#E53935', linewidth=2, markersize=8, label='Measured activation memory')
ax.plot(layer_counts, [expected_per_layer * n * 3 for n in layer_counts], '--', color='gray',
        linewidth=1.5, label=f'~{expected_per_layer*3:.0f} MB/layer (linear fit)')
ax.set_xlabel('Number of Layers', fontsize=12)
ax.set_ylabel('Activation Memory (MB)', fontsize=12)
ax.set_title('Activation Memory Grows Linearly with Depth', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("This is the problem: for deep models, activations consume most of the GPU memory.")
print("Activation checkpointing is our solution.")

In [ ]:
#@title 🎧 Listen: Checkpointing Scratch Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_checkpointing_scratch_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 2: Activation Checkpointing from Scratch

The key idea: instead of storing all $L$ activations, we only store activations at **checkpoint boundaries** and recompute the rest during the backward pass.

Let's implement this manually to understand exactly what happens under the hood.

In [ ]:
#@title 🎧 Code Walkthrough: Checkpointed Segment Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_checkpointed_segment_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class CheckpointedSegment(torch.autograd.Function):
    """
    A custom autograd function that implements activation checkpointing
    for a segment of layers.

    During the forward pass: runs the layers but does NOT save intermediate
    activations (only saves the input).

    During the backward pass: re-runs the forward pass to recompute
    activations, then computes gradients normally.
    """

    @staticmethod
    def forward(ctx, x, *layers):
        # Save the layers (not tensors, so we use ctx directly)
        ctx.layers = layers
        # Save the input for recomputation during backward
        ctx.save_for_backward(x.detach())

        # Run forward pass WITHOUT tracking gradients for intermediates
        with torch.no_grad():
            for layer in layers:
                x = layer(x)
        return x.detach().requires_grad_(True)

    @staticmethod
    def backward(ctx, grad_output):
        # Retrieve the saved input
        (x,) = ctx.saved_tensors
        x = x.detach().requires_grad_(True)

        # RECOMPUTE the forward pass (this is the extra compute cost)
        with torch.enable_grad():
            for layer in ctx.layers:
                x = layer(x)

        # Now compute gradients normally
        x.backward(grad_output)

        # Return gradients for the input (and None for each layer parameter)
        return (x.grad,) + (None,) * len(ctx.layers)


def run_with_manual_checkpointing(model_layers, x, segment_size):
    """
    Run a forward pass with manual activation checkpointing.

    Args:
        model_layers: list of nn.Module layers
        x: input tensor
        segment_size: number of layers per checkpoint segment
    """
    num_layers = len(model_layers)

    for start in range(0, num_layers, segment_size):
        end = min(start + segment_size, num_layers)
        segment = model_layers[start:end]
        x = CheckpointedSegment.apply(x, *segment)

    return x

print("Manual checkpointing implementation ready.")
print("Key insight: we only store the input to each segment, not every layer's output.")

In [ ]:
#@title 🎧 Narration: Verify Gradients
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_verify_gradients.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Let's verify our manual checkpointing produces the SAME gradients
# as standard backpropagation

torch.manual_seed(42)
hidden_dim = 256
num_layers = 8

# Create a simple model
layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(num_layers)]).to(device)
x = torch.randn(4, hidden_dim, device=device, requires_grad=True)

# Standard forward pass (store all activations)
x_standard = x.clone().detach().requires_grad_(True)
out = x_standard
for layer in layers:
    out = F.relu(layer(out))
loss_standard = out.sum()
loss_standard.backward()
grad_standard = x_standard.grad.clone()

# Checkpointed forward pass (our manual implementation)
x_ckpt = x.clone().detach().requires_grad_(True)

# Wrap each layer with ReLU for checkpointing
class LayerWithReLU(nn.Module):
    def __init__(self, linear):
        super().__init__()
        self.linear = linear
    def forward(self, x):
        return F.relu(self.linear(x))

wrapped_layers = [LayerWithReLU(l) for l in layers]
out_ckpt = run_with_manual_checkpointing(wrapped_layers, x_ckpt, segment_size=4)
loss_ckpt = out_ckpt.sum()
loss_ckpt.backward()
grad_ckpt = x_ckpt.grad.clone()

# Compare
max_diff = (grad_standard - grad_ckpt).abs().max().item()
print(f"Max gradient difference: {max_diff:.2e}")
print(f"Gradients match: {max_diff < 1e-5}")
print("\nCheckpointing produces identical gradients -- it is mathematically equivalent!")

del layers, x, x_standard, x_ckpt, out, out_ckpt
reset_memory_stats()

In [ ]:
#@title 🎧 Listen: Pytorch Checkpointing Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_pytorch_checkpointing_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 3: PyTorch's Built-in Checkpointing

Now that we understand the mechanism, let's use PyTorch's official `torch.utils.checkpoint.checkpoint` API. This is what you would use in practice -- it handles all the edge cases (RNG state, nested checkpointing, etc.).

In [ ]:
#@title 🎧 Code Walkthrough: Transformer Model Pytorch Ckpt
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_transformer_model_pytorch_ckpt.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class TransformerBlock(nn.Module):
    """A simplified transformer block for our experiments."""
    def __init__(self, hidden_dim, num_heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        # Self-attention with residual
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        # Feed-forward with residual
        x = x + self.ff(self.ln2(x))
        return x


class TransformerModel(nn.Module):
    """Transformer model with optional activation checkpointing."""
    def __init__(self, hidden_dim, num_layers, num_heads=8, use_checkpoint=False):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads) for _ in range(num_layers)
        ])
        self.use_checkpoint = use_checkpoint
        self.head = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpoint:
                # PyTorch's built-in checkpointing
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.head(x)

print("TransformerModel defined with optional checkpointing.")

In [ ]:
#@title 🎧 Narration: Compare Standard Vs Ckpt
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_compare_standard_vs_ckpt.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def measure_training_step(hidden_dim, num_layers, batch_size, seq_len, use_checkpoint):
    """
    Measure memory usage and time for one training step.

    Returns: (peak_memory_mb, step_time_seconds)
    """
    reset_memory_stats()

    model = TransformerModel(
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        use_checkpoint=use_checkpoint
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    x = torch.randn(batch_size, seq_len, hidden_dim, device=device)

    # Warm-up step
    optimizer.zero_grad()
    out = model(x)
    loss = out.sum()
    loss.backward()
    optimizer.step()

    # Reset stats after warm-up
    torch.cuda.reset_peak_memory_stats()

    # Measured step
    optimizer.zero_grad()
    torch.cuda.synchronize()
    start_time = time.perf_counter()

    out = model(x)
    loss = out.sum()
    loss.backward()
    optimizer.step()

    torch.cuda.synchronize()
    step_time = time.perf_counter() - start_time
    peak_mem = get_peak_memory_mb()

    del model, optimizer, x, out, loss
    reset_memory_stats()

    return peak_mem, step_time


# Compare memory usage: standard vs checkpointed
print("Comparing standard vs checkpointed training (12-layer transformer)")
print("=" * 70)

hidden_dim = 512
num_layers = 12
batch_size = 8
seq_len = 256

# Standard (no checkpointing)
mem_standard, time_standard = measure_training_step(
    hidden_dim, num_layers, batch_size, seq_len, use_checkpoint=False
)

# Checkpointed
mem_ckpt, time_ckpt = measure_training_step(
    hidden_dim, num_layers, batch_size, seq_len, use_checkpoint=True
)

print(f"\n{'Metric':<30} {'Standard':>12} {'Checkpointed':>14} {'Ratio':>10}")
print("-" * 70)
print(f"{'Peak memory (MB)':<30} {mem_standard:>12.1f} {mem_ckpt:>14.1f} {mem_ckpt/mem_standard:>10.2f}x")
print(f"{'Step time (ms)':<30} {time_standard*1000:>12.1f} {time_ckpt*1000:>14.1f} {time_ckpt/time_standard:>10.2f}x")
print(f"\nMemory savings: {(1 - mem_ckpt/mem_standard)*100:.1f}%")
print(f"Compute overhead: {(time_ckpt/time_standard - 1)*100:.1f}%")

In [ ]:
#@title 🎧 Listen: Scaling Experiment Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_scaling_experiment_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 4: Scaling Experiment -- How Memory Savings Grow with Depth

The theoretical memory reduction from checkpointing is $O(\sqrt{L})$ instead of $O(L)$. Let's verify this experimentally by measuring memory across different layer counts.

In [ ]:
#@title 🎧 Narration: Scaling Experiment Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_scaling_experiment_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Sweep over different layer counts
hidden_dim = 512
batch_size = 8
seq_len = 128
layer_counts = [4, 6, 8, 10, 12, 16, 20]

results_standard = []
results_ckpt = []

print(f"{'Layers':>8} {'Standard (MB)':>15} {'Checkpointed (MB)':>20} {'Savings':>10}")
print("-" * 60)

for n_layers in layer_counts:
    try:
        mem_s, _ = measure_training_step(hidden_dim, n_layers, batch_size, seq_len, False)
        mem_c, _ = measure_training_step(hidden_dim, n_layers, batch_size, seq_len, True)
        results_standard.append(mem_s)
        results_ckpt.append(mem_c)
        savings = (1 - mem_c / mem_s) * 100
        print(f"{n_layers:>8} {mem_s:>15.1f} {mem_c:>20.1f} {savings:>9.1f}%")
    except RuntimeError as e:
        print(f"{n_layers:>8}  OOM!")
        results_standard.append(None)
        results_ckpt.append(None)

In [ ]:
#@title 🎧 What to Look For: Scaling Experiment Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_scaling_experiment_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the memory scaling
valid = [(l, s, c) for l, s, c in zip(layer_counts, results_standard, results_ckpt) if s is not None]
if valid:
    ls, ss, cs = zip(*valid)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Absolute memory
    ax = axes[0]
    ax.plot(ls, ss, 'o-', color='#E53935', linewidth=2, markersize=8, label='Standard (all activations stored)')
    ax.plot(ls, cs, 's-', color='#43A047', linewidth=2, markersize=8, label='Checkpointed')
    ax.set_xlabel('Number of Layers', fontsize=12)
    ax.set_ylabel('Peak Memory (MB)', fontsize=12)
    ax.set_title('Peak Memory vs Model Depth', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    # Plot 2: Memory savings percentage
    ax = axes[1]
    savings_pct = [(1 - c/s)*100 for s, c in zip(ss, cs)]
    ax.bar(ls, savings_pct, color='#1E88E5', alpha=0.8, width=1.2)
    ax.set_xlabel('Number of Layers', fontsize=12)
    ax.set_ylabel('Memory Savings (%)', fontsize=12)
    ax.set_title('Memory Savings from Checkpointing', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

    print("\nKey observation: Memory savings increase with model depth.")
    print("For deep models (20+ layers), checkpointing saves 30-50% of peak memory.")

In [ ]:
#@title 🎧 Listen: Compute Tradeoff Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_compute_tradeoff_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 5: The Compute vs Memory Trade-off

Checkpointing is not free. We trade memory for compute. Let's quantify the overhead precisely.

In [ ]:
#@title 🎧 Narration: Compute Tradeoff Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_compute_tradeoff_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Measure compute overhead across different configurations
hidden_dim = 512
num_layers = 12
seq_len = 128
num_trials = 5

batch_sizes = [4, 8, 16]

print(f"{'Batch':>6} {'Standard (ms)':>15} {'Checkpoint (ms)':>17} {'Overhead':>10}")
print("-" * 55)

overhead_data = []

for bs in batch_sizes:
    times_s = []
    times_c = []

    for _ in range(num_trials):
        _, t_s = measure_training_step(hidden_dim, num_layers, bs, seq_len, False)
        _, t_c = measure_training_step(hidden_dim, num_layers, bs, seq_len, True)
        times_s.append(t_s)
        times_c.append(t_c)

    avg_s = np.median(times_s) * 1000
    avg_c = np.median(times_c) * 1000
    overhead = (avg_c / avg_s - 1) * 100
    overhead_data.append((bs, avg_s, avg_c, overhead))

    print(f"{bs:>6} {avg_s:>15.1f} {avg_c:>17.1f} {overhead:>9.1f}%")

print("\nTypical overhead: 20-40% (theory predicts ~33% for full checkpointing).")
print("This is the price we pay for the memory savings.")

In [ ]:
#@title 🎧 Listen: Selective Checkpointing Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_selective_checkpointing_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 6: Selective Checkpointing

Not all layers produce equally large activations. In a transformer:
- **Attention layers** produce activations of shape `(batch, heads, seq_len, seq_len)` -- this grows quadratically with sequence length!
- **Feed-forward layers** produce activations of shape `(batch, seq_len, 4*hidden_dim)` -- much smaller.

**Selective checkpointing** only checkpoints the memory-hungry layers (attention) and keeps the cheaper ones. This gives nearly the same memory savings with less compute overhead.

In [ ]:
#@title 🎧 Narration: Selective Checkpointing Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_selective_checkpointing_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SelectiveCheckpointTransformer(nn.Module):
    """Transformer with selective checkpointing -- only checkpoint attention."""
    def __init__(self, hidden_dim, num_layers, num_heads=8):
        super().__init__()
        self.blocks = nn.ModuleList()
        for _ in range(num_layers):
            self.blocks.append(nn.ModuleDict({
                'ln1': nn.LayerNorm(hidden_dim),
                'attn': nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True),
                'ln2': nn.LayerNorm(hidden_dim),
                'ff': nn.Sequential(
                    nn.Linear(hidden_dim, 4 * hidden_dim),
                    nn.GELU(),
                    nn.Linear(4 * hidden_dim, hidden_dim),
                ),
            }))
        self.head = nn.Linear(hidden_dim, hidden_dim)

    def _attn_forward(self, block, x):
        """Attention sub-block (this gets checkpointed)."""
        normed = block['ln1'](x)
        attn_out, _ = block['attn'](normed, normed, normed)
        return x + attn_out

    def forward(self, x):
        for block in self.blocks:
            # Checkpoint ONLY the attention (memory-heavy)
            x = checkpoint(self._attn_forward, block, x, use_reentrant=False)
            # Feed-forward runs normally (memory-light)
            x = x + block['ff'](block['ln2'](x))
        return self.head(x)


# Compare all three approaches
hidden_dim = 512
num_layers = 12
batch_size = 8
seq_len = 256

print("Comparing three approaches (12-layer transformer):")
print("=" * 75)

# 1. Standard (no checkpointing)
mem_s, time_s = measure_training_step(hidden_dim, num_layers, batch_size, seq_len, False)

# 2. Full checkpointing
mem_full, time_full = measure_training_step(hidden_dim, num_layers, batch_size, seq_len, True)

# 3. Selective checkpointing
reset_memory_stats()
model_sel = SelectiveCheckpointTransformer(hidden_dim, num_layers).to(device)
optimizer_sel = torch.optim.AdamW(model_sel.parameters(), lr=1e-4)
x_sel = torch.randn(batch_size, seq_len, hidden_dim, device=device)

# Warm up
optimizer_sel.zero_grad()
out_sel = model_sel(x_sel)
out_sel.sum().backward()
optimizer_sel.step()

torch.cuda.reset_peak_memory_stats()
optimizer_sel.zero_grad()
torch.cuda.synchronize()
start = time.perf_counter()
out_sel = model_sel(x_sel)
out_sel.sum().backward()
optimizer_sel.step()
torch.cuda.synchronize()
time_sel = time.perf_counter() - start
mem_sel = get_peak_memory_mb()

del model_sel, optimizer_sel, x_sel, out_sel
reset_memory_stats()

print(f"\n{'Approach':<25} {'Peak Memory (MB)':>18} {'Time (ms)':>12} {'Compute Overhead':>18}")
print("-" * 75)
print(f"{'Standard':<25} {mem_s:>18.1f} {time_s*1000:>12.1f} {'baseline':>18}")
print(f"{'Full Checkpointing':<25} {mem_full:>18.1f} {time_full*1000:>12.1f} {(time_full/time_s-1)*100:>17.1f}%")
print(f"{'Selective Checkpointing':<25} {mem_sel:>18.1f} {time_sel*1000:>12.1f} {(time_sel/time_s-1)*100:>17.1f}%")

print(f"\nSelective checkpointing: nearly as much memory savings, but less compute overhead!")

## Exercises


In [ ]:
#@title 🎧 Narration: Todo Exercise Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_todo_exercise_1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 1: Implement Segment-based Checkpointing

Instead of checkpointing every layer individually, group layers into segments of size $\sqrt{L}$ and checkpoint each segment. This is the approach described in the article.

For $L=16$ layers, create 4 segments of 4 layers each. Only the segment boundaries are checkpointed.

In [ ]:
#@title 🎧 Before You Start: Todo Exercise Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_todo_exercise_1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 1: Implement segment-based checkpointing
#
# Steps:
# 1. Create a model with 16 transformer blocks
# 2. Group them into sqrt(16) = 4 segments of 4 layers each
# 3. Wrap each SEGMENT (not individual layers) with checkpoint()
# 4. Compare memory usage with per-layer checkpointing

import math

class SegmentCheckpointModel(nn.Module):
    def __init__(self, hidden_dim, num_layers, num_heads=8):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads) for _ in range(num_layers)
        ])
        self.segment_size = int(math.sqrt(num_layers))
        self.head = nn.Linear(hidden_dim, hidden_dim)

    def _run_segment(self, layers_list, x):
        """Run a segment of layers sequentially."""
        for layer in layers_list:
            x = layer(x)
        return x

    def forward(self, x):
        # TODO: Split self.layers into segments of size self.segment_size
        # TODO: Wrap each segment with checkpoint()
        # Hint: Use checkpoint(self._run_segment, segment_layers, x, use_reentrant=False)
        pass  # Replace this with your implementation

# TODO: Measure memory for segment-based checkpointing
# Compare with: no checkpointing, per-layer checkpointing, segment checkpointing
# hidden_dim = 512, num_layers = 16, batch_size = 8, seq_len = 128

In [ ]:
#@title 🎧 Narration: Todo Exercise Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_todo_exercise_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 2: Find the Optimal Batch Size

With checkpointing enabled, you can fit larger batch sizes. Find the maximum batch size that fits in GPU memory for a 12-layer transformer, with and without checkpointing.

In [ ]:
#@title 🎧 Before You Start: Todo Exercise Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_todo_exercise_2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 2: Find maximum batch size
#
# Steps:
# 1. For a 12-layer transformer (hidden=512, seq_len=256):
#    a. Try increasing batch sizes [4, 8, 16, 32, 64, ...] WITHOUT checkpointing
#    b. Record which is the largest that fits without OOM
#    c. Repeat WITH checkpointing
# 2. Print the ratio: max_batch_ckpt / max_batch_standard
#
# Hint: Wrap the training step in a try/except RuntimeError to catch OOM

hidden_dim = 512
num_layers = 12
seq_len = 256
batch_sizes_to_try = [4, 8, 16, 32, 48, 64, 96, 128]

def find_max_batch(use_ckpt):
    max_bs = 0
    for bs in batch_sizes_to_try:
        try:
            # TODO: Call measure_training_step and check if it succeeds
            # If it does, update max_bs
            pass
        except RuntimeError:
            break
    return max_bs

# max_standard = find_max_batch(False)
# max_ckpt = find_max_batch(True)
# print(f"Max batch (standard): {max_standard}")
# print(f"Max batch (checkpointed): {max_ckpt}")
# print(f"Checkpointing allows {max_ckpt / max_standard:.1f}x larger batches!")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Summary

In this notebook, we have:

1. **Measured activation memory growth** -- confirmed it scales linearly with model depth $O(L)$

2. **Implemented checkpointing from scratch** -- understood the mechanism: save input, discard intermediates, recompute during backward

3. **Used PyTorch's `torch.utils.checkpoint`** -- the production-ready API that handles edge cases

4. **Quantified the trade-off** -- typically 30-50% memory savings for 20-35% more compute

5. **Explored selective checkpointing** -- checkpoint only the memory-hungry layers (attention) for better efficiency

### Key Takeaways

- Activations are the #1 memory consumer during training
- Checkpointing reduces activation memory from $O(L)$ to $O(\sqrt{L})$
- The compute overhead is ~33% (forward pass runs ~twice)
- Selective checkpointing is the practical choice: near-optimal memory savings with minimal overhead
- Checkpointing produces **mathematically identical** gradients -- no approximation

### Next Notebook

In Notebook 2, we will implement **gradient accumulation** -- the technique that lets us simulate large batch sizes without holding the full batch in memory.